In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/raw/credit_portfolio.csv"
)

In [3]:
defaults = df[
    df["default_flag"] == 1
].copy()

In [4]:
defaults.shape

(27431, 19)

In [5]:
defaults["lgd"].describe()

count    27431.000000
mean         0.551697
std          0.087546
min          0.168831
25%          0.492450
50%          0.552104
75%          0.610639
max          0.912751
Name: lgd, dtype: float64

In [6]:
defaults.groupby(
    pd.qcut(
        defaults["credit_score"],
        10
    )
)["lgd"].mean()

credit_score
(407.999, 579.0]    0.547670
(579.0, 606.0]      0.554232
(606.0, 625.0]      0.545865
(625.0, 641.0]      0.553021
(641.0, 656.0]      0.555202
(656.0, 671.0]      0.554940
(671.0, 687.0]      0.550158
(687.0, 706.0]      0.553187
(706.0, 732.0]      0.550967
(732.0, 850.0]      0.551622
Name: lgd, dtype: float64

In [7]:
defaults.groupby(
    pd.qcut(
        defaults["debt_to_income"],
        10
    )
)["lgd"].mean()

debt_to_income
(0.009000000000000001, 0.125]    0.526510
(0.125, 0.184]                   0.536123
(0.184, 0.234]                   0.536624
(0.234, 0.281]                   0.543329
(0.281, 0.329]                   0.547522
(0.329, 0.378]                   0.552513
(0.378, 0.433]                   0.556948
(0.433, 0.496]                   0.563604
(0.496, 0.583]                   0.570370
(0.583, 0.941]                   0.583432
Name: lgd, dtype: float64

In [8]:
lgd_target = defaults["lgd"]

In [9]:
lgd_features = [
    "credit_score",
    "debt_to_income",
    "credit_utilization",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "previous_defaults",
    "delinquencies_12m"
]

In [10]:
defaults["observation_date"] = pd.to_datetime(
    defaults["observation_date"]
)

lgd_train = defaults[
    defaults["observation_date"]
    < "2023-07-01"
].copy()

lgd_val = defaults[
    (
        defaults["observation_date"]
        >= "2023-07-01"
    )
    &
    (
        defaults["observation_date"]
        < "2023-10-01"
    )
].copy()

lgd_oot = defaults[
    defaults["observation_date"]
    >= "2023-10-01"
].copy()

In [11]:
X_lgd_train = lgd_train[
    lgd_features
]

y_lgd_train = lgd_train[
    "lgd"
]

X_lgd_val = lgd_val[
    lgd_features
]

y_lgd_val = lgd_val[
    "lgd"
]

X_lgd_oot = lgd_oot[
    lgd_features
]

y_lgd_oot = lgd_oot[
    "lgd"
]

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

lgd_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [14]:
from sklearn.ensemble import (
    RandomForestRegressor
)

lgd_model = Pipeline(
    steps=[
        (
            "preprocessor",
            lgd_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=30,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [15]:
lgd_model.fit(
    X_lgd_train,
    y_lgd_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['credit_score','debt_to_income','credit_utilization',...,'interest_rate', 'previous_defaults','delinquencies_12m']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformer

In [26]:
import joblib
from pathlib import Path

models_dir = Path("../models")

models_dir.mkdir(
    exist_ok=True
)

joblib.dump(
    lgd_model,
    models_dir / "lgd_model.joblib"
)

['..\\models\\lgd_model.joblib']

In [18]:
lgd_predictions = (lgd_model.predict(
    X_lgd_oot
)
                  )

In [19]:
lgd_predictions = (
    lgd_model.predict(
        X_lgd_oot
    )
)

In [20]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [21]:
lgd_mae = mean_absolute_error(
    y_lgd_oot,
    lgd_predictions
)

lgd_rmse = np.sqrt(
    mean_squared_error(
        y_lgd_oot,
        lgd_predictions
    )
)

In [22]:
print(
    f"LGD MAE: {lgd_mae:.4f}"
)

print(
    f"LGD RMSE: {lgd_rmse:.4f}"
)

LGD MAE: 0.0645
LGD RMSE: 0.0809


In [23]:
lgd_comparison = pd.DataFrame({
    "actual_lgd": y_lgd_oot.values,
    "predicted_lgd": lgd_predictions
})

lgd_comparison.head()

,actual_lgd,predicted_lgd
0,0.594538,0.573667
1,0.437028,0.527408
2,0.532752,0.553342
3,0.609197,0.555725
4,0.448235,0.494201


In [24]:
print(
    "Actual average LGD:",
    lgd_comparison[
        "actual_lgd"
    ].mean()
)

print(
    "Predicted average LGD:",
    lgd_comparison[
        "predicted_lgd"
    ].mean()
)

Actual average LGD: 0.5500045400854563
Predicted average LGD: 0.5529870250766283


In [25]:
lgd_comparison.to_csv(
    "../data/outputs/lgd_predictions.csv",
    index=False
)